## Gensim

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\Karan\Downloads\IMDB Dataset.csv")
df.head(5)
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [4]:
df = df.dropna(subset=['review','sentiment'])
df = df.drop_duplicates()
df.shape

(49582, 2)

In [7]:
import nltk

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Karan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Karan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Karan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re

stop_words = set(stopwords.words('english'))
lm = WordNetLemmatizer()

def preprocess(txt):
    txt = txt.lower()
    txt = re.sub(r'<[^>]*>','',txt)
    txt = re.sub(r'[^a-z\s]', '', txt)
    tokens = word_tokenize(txt)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lm.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['tokens'] = df['review'].apply(preprocess)


In [ ]:
x_text = df['tokens']
y_text = df['sentiment']

from sklearn.model_selection import train_test_split

x_train_text, x_test_text, y_train_text, y_test_text = train_test_split(x_text,y_text, test_size=0.2,random_state=42, stratify=y_text)

In [16]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec(sentences=x_train_text,vector_size=100,window=5,min_count=2,workers=4,sg=1,epochs=10)

In [17]:
print('Vocabulary Size:', len(word2vec_model.wv))

Vocabulary Size: 27


In [18]:
import numpy as np

def document_vector(tokens,model):
    vectors=[]
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    if len(vectors) == 0:
        return np.zeros(model,vector_size)
    return np.mean(vectors,axis=0)

In [19]:
x_train = np.array([document_vector(tokens,word2vec_model) for tokens in x_train_text])
x_test = np.array([document_vector(tokens,word2vec_model) for tokens in x_test_text])


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_text)
y_test_encoded = le.transform(y_test_text)

lm = LogisticRegression(max_iter=1000)
lm.fit(x_train,y_train_encoded)
y_pred = lm.predict(x_test)

acc = accuracy_score(y_test_encoded,y_pred)
acc

0.584350105878794